# 3. Capacity, optimization, and data budgets

**Question:** Is a locality effect explained by finite network capacity, too few optimization updates, or limited training observations?

This workflow now studies those mechanisms; its historical filename is retained for existing links. Noise and aliasing comparisons are in workflow 4.

Compare linear encoders and MLP widths 32, 64, and 256 without changing output dimensionality. Evaluate several checkpoints along the **same training run**, then vary data budget in a separate slice instead of crossing every factor. Equal update budgets and equal epochs answer different questions; neither alone proves asymptotic sample complexity.

No cross-modal training targets are introduced. Outputs remain in RAM unless saving is explicitly enabled. Your editor may still save displayed output inside the notebook itself.

In [ ]:
from pathlib import Path
import sys
import subprocess
from zipfile import ZipFile

REPO_URL = 'https://github.com/AlexM3020/The-Role-of-Dynamical-Structure-in-Cross-Modal-Representation-Learning.git'
REPO_REF = 'main'  # For a frozen run, choose a commit before starting a fresh runtime.
SOURCE_ARCHIVE = None  # Optional uploaded source ZIP; not needed in Colab with GitHub access.
INSTALL_DEPENDENCIES = False  # Enable only if the runtime lacks compatible dependencies.
cloud = next((p for p in (Path('/content'), Path('/kaggle/working')) if p.exists()), None)
if SOURCE_ARCHIVE is not None:
    if cloud is None:
        raise RuntimeError('Use the local checkout here; source extraction is for cloud runtimes.')
    destination = cloud / 'dynalign_uploaded_source'
    with ZipFile(SOURCE_ARCHIVE) as archive:
        for name in archive.namelist():
            if not (destination / name).resolve().is_relative_to(destination.resolve()):
                raise ValueError('Unsafe archive member')
        archive.extractall(destination)
    candidates = [destination]
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / 'src/dynalign/revised.py').exists()), None)
if ROOT is None and SOURCE_ARCHIVE is None and cloud is not None:
    destination = cloud / 'dynalign_github'
    if not destination.exists():
        subprocess.check_call(['git', 'clone', '--quiet', REPO_URL, str(destination)])
        subprocess.check_call(['git', '-C', str(destination), 'fetch', '--quiet', 'origin', REPO_REF])
        subprocess.check_call(['git', '-C', str(destination), 'checkout', '--quiet', '--detach', 'FETCH_HEAD'])
    ROOT = destination
if ROOT is None or not (ROOT / 'src/dynalign/revised.py').exists():
    raise FileNotFoundError('Open the current checkout locally, or use a fresh Colab/Kaggle runtime with Internet enabled.')
# Existing checkouts are deliberately reused, not reset or silently updated.
# Restart in a fresh runtime to switch REPO_REF or load a newer GitHub revision.
if 'dynalign' in sys.modules and Path(sys.modules['dynalign'].__file__).resolve().parents[2] != ROOT.resolve():
    raise RuntimeError('Another source version is already imported; restart the runtime before switching.')
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', str(ROOT)])
sys.path.insert(0, str(ROOT / 'src'))
revision = subprocess.run(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], capture_output=True, text=True)
print('Source revision:', revision.stdout.strip() if revision.returncode == 0 else 'uploaded source bundle')
print('Source setup only: result saving remains disabled by default.')

from dataclasses import replace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dynalign.revised import FitSettings, EvalSettings, SyntheticSpec, run_synthetic, preview_plan, paired_contrasts, bundle_zip

## Experiments to try

1. **Capacity slice (`SLICE='capacity'`):** hold 2,048 observations fixed; compare a linear encoder and MLP widths 32 / 64 / 256, output dimension four. Start with 10 independent worlds and two repeats after the execution check.
2. **Optimization checkpoints:** evaluate 800 and 3,200 updates from the same training trajectory. The later checkpoint is not a new random initialization. Do not select checkpoints by test CKA.
3. **Data slice (`SLICE='data'`):** fix width 64; compare 512 / 2,048 / 8,192 transitions. Use 25 new worlds for the primary comparison. Keep the chosen objective fixed.
4. **Objective sensitivity:** compare the original regime with the candidate frozen from the loss investigation. The default identity penalty below is a candidate, not an already selected winner.

More capacity can change optimization and approximation simultaneously. A disappearing gap is evidence of sensitivity, not proof of one exclusive cause. Report measured time, parameter counts, update count, data reuse, rank and alignment together. For the same seed, different budgets are paired but not automatically nested training datasets.

In [ ]:
RUN_TRAINING = False
SLICE = 'capacity'  # 'capacity' or 'data': avoid an unnecessary full Cartesian grid.
PHASE = 'development'
WORLD_IDS, REPEAT_IDS = [200], [0]
MASTER_SEED = 20261108
CONDITIONS = ['local', 'scrambled']
CAPACITY_MODELS = [('linear', 64), ('mlp', 32), ('mlp', 64), ('mlp', 256)]
CAPACITY_DATA_BUDGET = 256  # Expanded: 2048
DATA_MODELS = [('mlp', 64)]
DATA_BUDGETS = [256, 512]  # Expanded: [512, 2048, 8192]
N_STATES, SENSOR_WIDTH, EMBEDDING_DIM, NOISE = 32, 12, 4, 0.05
DEVICE, CPU_THREADS = 'auto', 2
SAVE_RESULTS, OUTPUT_DIR = False, None
FIT = FitSettings(steps=80, checkpoints=(40, 80), batch_size=128, lr=1e-3,
                  weight_decay=1e-4, invariance_weight=25, variance_weight=25,
                  covariance_weight=1, identity_weight=25, variance_target=1,
                  epsilon=1e-4, clip_grad=5, log_every=40, checkpoint_every=100)
# Expanded trajectory: steps=3200, checkpoints=(800, 3200). Both conditions get both checkpoints.
EVAL = EvalSettings(calibration_repeats=8, test_repeats=16, ridge=1e-3,
                    collapse_variance=1e-6, collapse_rank_fraction=0.5, batch_size=512)
REGIMES = {
    'baseline': dict(objective='vicreg', invariance_weight=25, variance_weight=25, covariance_weight=1),
    'identity_candidate': dict(objective='cov_identity', invariance_weight=25, identity_weight=25),
}
ACTIVE_REGIMES = ['baseline', 'identity_candidate']  # Replace candidate with the frozen development choice.
if SLICE not in {'capacity', 'data'}:
    raise ValueError('Choose capacity or data')
if SAVE_RESULTS and OUTPUT_DIR is None:
    raise ValueError('Saving requires an explicit destination.')
models = CAPACITY_MODELS if SLICE == 'capacity' else DATA_MODELS
budgets = [CAPACITY_DATA_BUDGET] if SLICE == 'capacity' else DATA_BUDGETS
specs = [SyntheticSpec(experiment='capacity_' + SLICE, phase=PHASE, world=w, repeat=r,
                      master_seed=MASTER_SEED, condition=c, n_states=N_STATES, n_pairs=n,
                      sensor_width=SENSOR_WIDTH, noise=NOISE, architecture=architecture,
                      width=width, dim=EMBEDDING_DIM, regime=regime,
                      fit=replace(FIT, **REGIMES[regime]), evaluation=EVAL)
         for w in WORLD_IDS for r in REPEAT_IDS for n in budgets for architecture, width in models
         for regime in ACTIVE_REGIMES for c in CONDITIONS]
display(preview_plan(specs))
print(f'{2 * len(specs)} encoder fits, with multiple checkpoints per fit. Training enabled: {RUN_TRAINING}')

## Training trajectories, not independent checkpoint replications

The larger update count continues the smaller checkpoint's trajectory. Calibrated alignment and probes are recomputed at each requested checkpoint without fitting on the test observations or changing the encoder. Recorded runtime includes these evaluations and is not pure GPU kernel time.

**Loss settings in `REGIMES` override the corresponding `FIT` defaults; edit the regime dictionary for a loss ablation.** Other optimizer settings come from `FIT`, and the displayed plan shows the effective merged values.

To compare capacities fairly, pair conditions within each architecture and record exact parameter counts. The linear baseline need not have the same number of parameters as an MLP. Keep all other settings identical within a comparison.

In [ ]:
bundle = None
if RUN_TRAINING:
    bundle = run_synthetic(specs, device=DEVICE, threads=CPU_THREADS,
                           output_dir=OUTPUT_DIR if SAVE_RESULTS else None)
    results = bundle.table
    display(results[['regime', 'architecture', 'width', 'condition', 'world', 'n_pairs',
                     'step', 'cka', 'any_collapse', 'parameters_a', 'parameters_b',
                     'run_seconds_including_evaluation']])
else:
    print('Plan inspected; training remains disabled until RUN_TRAINING=True.')

## Interpret capacity and optimization separately

Group contrasts by architecture, width, regime, data budget, and checkpoint. Seeds at different checkpoints are repeated observations of one training run, not extra independent worlds.

- If the gap shrinks with updates, optimization budget is relevant.
- If it shrinks with width, capacity/optimization interaction is relevant.
- If it remains, the result is more robust within the tested settings, not universal.

Show rank and spectral recovery as well as CKA. Undefined values and collapse must remain visible. The same 25 development worlds must not be relabeled as independent confirmation.

In [ ]:
if bundle is not None:
    contrasts = paired_contrasts(results, metric='cka',
                                group_by=['architecture', 'width', 'regime', 'n_pairs', 'step'])
    display(contrasts)
    display(results[['architecture', 'width', 'regime', 'condition', 'n_pairs', 'step',
                     'effective_rank_a', 'effective_rank_b', 'any_collapse']])
    fig, ax = plt.subplots(figsize=(10, 4))
    for (architecture, width, regime, n, condition), block in results.groupby(['architecture', 'width', 'regime', 'n_pairs', 'condition']):
        means = block.groupby('step').cka.mean()
        ax.plot(means.index, means.values, 'o-', label=f'{architecture}{width}/{regime}/N={n}/{condition}')
    ax.set(xlabel='Optimizer updates', ylabel='Mean CKA', title='Checkpoints along each training run')
    ax.legend(fontsize=6, bbox_to_anchor=(1, 1))
    fig.tight_layout()
    plt.show()
else:
    print('No training executed. Choose the capacity or data slice before enabling it.')

## Optional export

The export is prepared in RAM and contains the complete plan, including loss weights, architecture, optimizer and update checkpoints. No results are written unless you explicitly enabled saving and supplied a directory. With memory-only execution, keep the runtime alive or deliberately export before it ends.

Showing a download link is optional; clicking it is a separate manual action that can save to your laptop. Your notebook editor may also retain displayed cell outputs when you save the notebook. Neither behavior is used as automatic experiment persistence.

In [ ]:
SHOW_DOWNLOAD_LINK = False
EXPORT_BYTES = bundle_zip(bundle) if bundle is not None else None
if EXPORT_BYTES is not None:
    print(f'{len(EXPORT_BYTES):,} export bytes in RAM. Saving enabled: {SAVE_RESULTS}')
    if SHOW_DOWNLOAD_LINK:
        import base64
        from IPython.display import HTML
        display(HTML('<a download="capacity_optimization.zip" href="data:application/zip;base64,' +
                     base64.b64encode(EXPORT_BYTES).decode() + '">Click to download</a>'))
else:
    print('No results to export. No file or browser download was created.')